# Notebook File Overview

In this notebook, we will be importing the data frames we created in the first two notebooks:
1. person_control.csv
2. person_study.csv
3. survey_control.csv
4. survey_study.csv
5. measurement_control.csv
6. measurement_study.csv

We will use the code snippet provided by *All of Us* to import our data frames from the workspace bucket (looks similar to the snippet used to save our data frames to the workspace bucket). 

We will do some data processing and cleaning on our existing data frames including:
* Sort both cohorts by person_ID
* Clean up answer and question columns
* remove duplicate person_IDs
* Create a new variable to designate which participants **have** and **do not have** diabetes (flagging varaible) 
* Combine the two dataframes into one dataframe
* Calculate participant age based on the current date and their DOB and create a new variable called *age*
* Create new variable called *age_group* that puts age into 10 yuear buckets
* Select the final columns we want for our final dataframe we will analyze

# Add the code snippet to copy data *FROM* the *All of Us R and Cloud Storage snippets*

**Step 1: Run *Setup***

In [ ]:
library(tidyverse)  # Data wrangling packages.

**Step 2: Run the *copy_file_from_workspace_bucket.R* code snippet**

This will import:

* person_control.csv
* person_study.csv
* survey_control.csv
* survey_study.csv
* measurement_control.csv
* measurement_study.csv

as data frames

**NOTE: You will have to add another line of code to each step to get both files imported**

<div style=" background-color:#f8d7da; color:#842029; border:1px solid #f5c2c7; border-radius:8px; padding:20px; font-size:20px; "> 
<b>We are going to update the provided snippet, even past the <i>DON'T CHANGE FROM HERE</i> Warning</b><br> 
</div>

In [ ]:
# This snippet assumes that you run setup first

# This code copies a file from your Google Bucket into a dataframe

# replace 'test.csv' with the name of the file in your google bucket (don't delete the quotation marks)

## CHANGE ORIGINAL TO THIS 
## ORIGINAL: name_of_file_in_bucket <- 'test.csv'
person_control <- 'person_control.csv'
person_study <- 'person_study.csv'
survey_control <- 'survey_control.csv'
survey_study <- 'survey_study.csv'
measurement_control <- 'measurement_control.csv'
measurement_study <- 'measurement_study.csv'

########################################################################
##
################# DON'T CHANGE FROM HERE ###############################
##
########################################################################

# Get the bucket name
my_bucket <- Sys.getenv('WORKSPACE_BUCKET')

# Copy the file from current workspace to the bucket

## CHANGE ORIGINAL TO THIS 
## ORIGINAL: system(paste0("gsutil cp ", my_bucket, "/data/", name_of_file_in_bucket, " ."), intern=T)
system(paste0("gsutil cp ", my_bucket, "/data/", person_control, " ."), intern=T)
system(paste0("gsutil cp ", my_bucket, "/data/", person_study, " ."), intern=T)
system(paste0("gsutil cp ", my_bucket, "/data/", survey_control, " ."), intern=T)
system(paste0("gsutil cp ", my_bucket, "/data/", survey_study, " ."), intern=T)
system(paste0("gsutil cp ", my_bucket, "/data/", measurement_control, " ."), intern=T)
system(paste0("gsutil cp ", my_bucket, "/data/", measurement_study, " ."), intern=T)

# Load the file into a dataframe

## CHANGE ORIGINAL TO THIS 
## ORIGINAL: my_dataframe  <- read_csv(name_of_file_in_bucket)
person_control  <- read_csv(person_control)
person_study  <- read_csv(person_study)
survey_control  <- read_csv(survey_control)
survey_study  <- read_csv(survey_study)
measurement_control  <- read_csv(measurement_control)
measurement_study  <- read_csv(measurement_study)


## CHANGE ORIGINAL TO THIS 
## ORIGINAL: head(my_dataframe)
head(person_control)
head(person_study)
head(survey_control)
head(survey_study)
head(measurement_control)
head(measurement_study)

# Let's clean and join our data before saving to our workspace bucket

**Step 1: Sort our survey_control, survey_study data, measurement_control, measurement_study by *person_id***

Create a new object with that includes "arranged" in the name using the assignment (<-) operator

Then, we will use the *arrange()* function from the **Dplyr** package within the **Tidyverse** (loaded earlier)

In [ ]:
#Survey data
survey_control_arranged <- survey_control %>%
arrange(person_id)

survey_study_arranged <- survey_study %>%
arrange(person_id)

#Measurement data
measurement_control_arranged <- measurement_control %>%
arrange(person_id)

measurement_study_arranged <- measurement_study %>%
arrange(person_id)

In [ ]:
#Survey data
dim(survey_control_arranged)
head(survey_control_arranged, 5)

dim(survey_study_arranged)
head(survey_study_arranged, 5)

#Measurement data
dim(measurement_control_arranged)
head(measurement_control_arranged, 5)

dim(measurement_study_arranged)
head(measurement_study_arranged, 5)

## Let's start with survey data 

**Step 1: Clean up the answer columns**

We will create new objects called survey_control_clean and survey_study_clean which are modified dataframes from the previous survey_control_arranged and survey_study_arranged dataframes

As you can see in the previous chunk, the answer column includes the same string of text found in the question column which we want to **remove**

We will use the *str_remove()* and *str_trim()* functions from the **stringr** package within the **Tidyverse**

In [ ]:
survey_control_clean <- survey_control_arranged %>%
  mutate(
    answer = str_remove(answer, fixed(question)), # remove the exact question text (literal match)
    answer = str_remove(answer, "^\\s*-\\s*"),    # remove a leading hyphen and surrounding spaces left behind
    answer = str_trim(answer)                     # trim any leftover whitespace
  )


survey_study_clean <- survey_study_arranged %>%
  mutate(
    answer = str_remove(answer, fixed(question)), # remove the exact question text (literal match)
    answer = str_remove(answer, "^\\s*-\\s*"),    # remove a leading hyphen and surrounding spaces left behind
    answer = str_trim(answer)                     # trim any leftover whitespace
  )

**Step 2: Make the values in the question column the column names and the values in the answer column the values for the new columns**

This will get rid of the person_id duplicates but keep both the questions and their responses

We will create a new object called survey_wide which is modifies the previous survey_clean data frame

We will use the *pivot_wider()* function from the **tidyr** package within the **Tidyverse** to accomplish this

In [ ]:
survey_control_wide <- survey_control_clean %>%
  pivot_wider(
    names_from = question,
    values_from = answer
  )

survey_study_wide <- survey_study_clean %>%
  pivot_wider(
    names_from = question,
    values_from = answer
  )

In [ ]:
dim(survey_control_wide)
head(survey_control_wide, 5)

dim(survey_study_wide)
head(survey_study_wide, 5)

**Step 4: Join the updated survey dataframes (survey_control_wide and survey_study_wide), to their respective demographic variable dataframe (person_control and person_study)**

We will create two new objects called *diabetes_control_ptsd* and *diabetes_control_ptsd* which will include variables from our survey and demographic variable dataframes.

We will use the *left_join()* function from the **dplyr** package to join the two data frames on the **person_id** variable.

We will also use the *filter()* function from the **dplyr** package to remove NA values from our new data frame.

In [ ]:
library(dplyr)

diabetes_control_ptsd <- person_control %>%
  left_join(survey_control_wide, by = c("person_id" = "person_id")) %>%
  filter(
    !is.na(`Are you still seeing a doctor or health care provider for post-traumatic stress disorder (PTSD)?`),
    !is.na(`Are you currently prescribed medications and/or receiving treatment for post-traumatic stress disorder (PTSD)?`)
  )

diabetes_study_ptsd <- person_study %>%
  left_join(survey_study_wide, by = c("person_id" = "person_id")) %>%
  filter(
    !is.na(`Are you still seeing a doctor or health care provider for post-traumatic stress disorder (PTSD)?`),
    !is.na(`Are you currently prescribed medications and/or receiving treatment for post-traumatic stress disorder (PTSD)?`)
  )

dim(diabetes_control_ptsd)
head(diabetes_control_ptsd, 5)

dim(diabetes_study_ptsd)
head(diabetes_study_ptsd, 5)

**Step 5: Add new columns to each data frame called *diabetes* and include values based on diabetes status**

We will use the mutate() function from the **dplyr** package to create new variables for each data frame.

In [ ]:
library(dplyr)

diabetes_no_ptsd <- diabetes_control_ptsd %>%
  mutate(diabetes = "No")

diabetes_yes_ptsd <- diabetes_study_ptsd%>%
  mutate(diabetes = "Yes")

dim(diabetes_no_ptsd)
head(diabetes_no_ptsd)
dim(diabetes_yes_ptsd)
head(diabetes_yes_ptsd)

**Step 6: combine the survey dataframes**

We will use several functions from the **dplyr** package to create our final analysis-ready dataset:

* Use the *bind_rows()* function to combine the diabetes_no_ptsd and diabetes_yes_ptsd data frames into one dataset
* Use the *rename()* function to give shorter, more manageable names to the long PTSD survey question columns
* Use the *mutate()* function to calculate participant age from their date of birth and create a new variable called *age*
* Use the *select()* function to reorder columns and remove the original date_of_birth column since we now have the calculated age


We use bind_rows() instead of a join function because the column names for each data frame are the same and in the same order, so we can simply stack the data together which is easier, cleaner, and has fewer potential errors

In [ ]:
final_diabetes_ptsd <- bind_rows(diabetes_no_ptsd, diabetes_yes_ptsd) %>%
  rename(
    ptsd_doctor = `Are you still seeing a doctor or health care provider for post-traumatic stress disorder (PTSD)?`,
    ptsd_treatment = `Are you currently prescribed medications and/or receiving treatment for post-traumatic stress disorder (PTSD)?`
  ) %>%
  mutate(
    age = floor(interval(ymd_hms(date_of_birth), today()) / years(1))
  ) %>%
select(person_id, age, gender, race, ethnicity, diabetes, everything(), -date_of_birth, -sex_at_birth)


dim(final_diabetes_ptsd)
head(final_diabetes_ptsd)
#str(final_diabetes_ptsd)

## Now let's clean up the measurement data and join it to our final DF

**Step 1: clean up the values in 'standard concept name' by combining value into just a few categories**

In [ ]:
# run this to see what the values are for each measurement DF
measurement_control_arranged %>%
  distinct(standard_concept_name, measurement_concept_id)

measurement_study_arranged %>%
  distinct(standard_concept_name, measurement_concept_id)

# PRO TIP!
# Copy/paste the list into an AI chatbot and use a prompt like: "I need to combine these into just a few related categories: {paste list}"
# ask to provide helper function sthat we can apply to both DFs

**Step 2: Add the helper function to each DF**

In [ ]:
library(dplyr)
library(stringr)

# -------- helper function -------------------------------------------
collapse_measure <- function(name_vec) {
  case_when(
    # BMI
    str_detect(name_vec, regex("body mass index", ignore_case = TRUE)) ~ "BMI",
    
    # Systolic blood pressure (sitting, standing, ambulatory, lying, generic)
    str_detect(name_vec, regex("systolic.*blood pressure", ignore_case = TRUE)) ~ "Systolic_BP",
    
    # HbA1c / glycated hemoglobin
    str_detect(name_vec, regex("hemoglobin a1c|hba1c|glycated hemoglobin", ignore_case = TRUE)) ~ "HbA1c",
    
    # LDL-related cholesterol (any wording, any method, LDL/HDL ratio as LDL group)
    str_detect(name_vec, regex("cholesterol.*ldl", ignore_case = TRUE)) ~ "LDL_Cholesterol",
    
    TRUE ~ "Other"   # fallback, should be rare/empty
  )
}

# -------- apply to each data frame ----------------------------------
measurement_control_cats <- measurement_control_arranged %>% 
  mutate(measure_group = collapse_measure(standard_concept_name))

measurement_study_cats <- measurement_study_arranged %>% 
  mutate(measure_group = collapse_measure(standard_concept_name))

# -------- inspect results ---------------------------------
measurement_control_cats %>% count(measure_group, sort = TRUE)
measurement_study_cats   %>% count(measure_group, sort = TRUE)

**Step 3: Let's add some code to keep only the most recent measurement for each value type**

In [ ]:
#head(measurement_control_cats, 5)
#head(measurement_study_cats, 5)

library(dplyr)
library(lubridate)   # for ymd_hms()

measurement_control_latest <- measurement_control_cats %>% 
  arrange(person_id, measure_group, desc(measurement_datetime)) %>% 
  group_by(person_id, measure_group) %>% 
  slice_head(n = 1) %>%
  ungroup()

measurement_study_latest <- measurement_study_cats %>% 
  arrange(person_id, measure_group, desc(measurement_datetime)) %>% 
  group_by(person_id, measure_group) %>% 
  slice_head(n = 1) %>% 
  ungroup()

head(measurement_control_latest, 5)
head(measurement_study_latest, 5)

**Step 4: Make the values in the measure_group column the column names and the values in the value_as_number column the values for the new columns**

This will get rid of the person_id duplicates but keep both the measurement and their values

We will create a new objects called measurement_control_wide and measurement_study_wide which modifies the previous measurement_control_latest and measurement_study_latest data frames

We will use the *pivot_wider()* function from the **tidyr** package within the **Tidyverse** to accomplish this

In [ ]:
measurement_control_wide <- measurement_control_latest %>% 
  pivot_wider(
    id_cols     = person_id,           # one row per person
    names_from  = measure_group,       # BMI, Systolic_BP, HbA1c, LDL_Cholesterol
    values_from = value_as_number
  )

measurement_study_wide <- measurement_study_latest %>% 
  pivot_wider(
    id_cols     = person_id,
    names_from  = measure_group,
    values_from = value_as_number
  )

dim(measurement_control_wide) # N was 4,651,005 now N is 114,229
head(measurement_control_wide, 5)
dim(measurement_study_wide) # N was 1,432,768 now N is 8,098
head(measurement_study_wide,   5)

**Step 5: combine the measurement dataframes**

We will use the bind_rows() functions from the **dplyr** package to combine our measurement_control_wide and measurement_study_wide dataframes to create our final_measurements dataframe:

We use bind_rows() instead of a join function because the column names for each data frame are the same and in the same order, so we can simply stack the data together which is easier, cleaner, and has fewer potential errors

In [ ]:
final_measurement <- bind_rows(measurement_control_wide, measurement_study_wide)

dim(final_measurements)
head(final_measurements)
#str(final_measurements)

## Step 5: Join the final_measurement and final_diabetes_ptsd dataframes to create our final_analysis dataframe

This final dataset will include personID, demographic variables, and measurement varaibles with the diabetes flag varaibles we created

In [ ]:
library(dplyr)

final_analysis <- final_diabetes_ptsd %>%
  left_join(final_measurement, by = c("person_id" = "person_id")) %>%
  drop_na() %>%
  unique()

dim(final_analysis)
head(final_analysis, 5)

## Now, we are going to do some finally cleaning before savign the two DFs so they are analysis ready

- We are going to drop `PMI: Skip` from ptsd_doctor and ptsd_treatment within final_diabetes_ptsd DF
- We are going to create a new varaible called *age_groups* within final_measurements DF
    - **Note: our age range for our cohort is 25 to 50 years old**

In [ ]:
# Update final_diabetes_ptsd
final_analysis <- final_analysis %>%
  filter(ptsd_doctor != "PMI: Skip" & ptsd_treatment != "PMI: Skip") %>%
  mutate(
    age_group = cut(
      age,
      breaks = c(25, 35, 45, 50),
      labels = c("25-34", "35-44", "45-49"),
      right = FALSE,
      include.lowest = TRUE
    )
  )

table(final_analysis$gender)
table(final_analysis$ptsd_treatment)
head(final_analysis)

# Add the code snippet to copy data from the *All of Us R and Cloud Storage snippets*

<div class="alert alert-block alert-info" style="font-size:16px">
    
<b>Navigate to the code snippets</b>
    <ul style="margin-top: 0px;">    
        <li>All of Us R and Cloud Storage snippets -> Copy file to or from Workspace Bucket -> copy_data_to_workspace_bucket.R</li>
    </ul>
    
<b>Read the code carefully</b>
    <ul style="margin-top: 0px;">    
        <li>Replace <b>df</b> with <i>THE NAME OF YOUR DATAFRAME</i>
            <ul>
                <li><i>final_diabetes_ptsd</i></li>
            </ul>
        </li>
        <li>Replace <b>'test.csv'</b> with <i>THE NAME of the file</i> you're going to store in the bucket (don't delete the quotation marks)
            <ul>
                <li><i>'final_diabetes_ptsd.csv'</i></li>
            </ul>
        </li>
    </ul>
        
</div>

In [ ]:
# This snippet assumes that you run setup first

# This code saves your dataframe into a csv file in a "data" folder in Google Bucket

# Replace df with THE NAME OF YOUR DATAFRAME
final_analysis <- final_analysis

# Replace 'test.csv' with THE NAME of the file you're going to store in the bucket (don't delete the quotation marks)
final_analysis_filename <- 'final_analysis.csv'

########################################################################
##
################# DON'T CHANGE FROM HERE ###############################
##
########################################################################

# store the dataframe in current workspace
write_excel_csv(final_analysis, final_analysis_filename)

# Get the bucket name
my_bucket <- Sys.getenv('WORKSPACE_BUCKET')

# Copy the file from current workspace to the bucket
system(paste0("gsutil cp ./", final_analysis_filename, " ", my_bucket, "/data/"), intern=T)

# Check if file is in the bucket
system(paste0("gsutil ls ", my_bucket, "/data/*.csv"), intern=T)


In [ ]:
library(bigrquery)
library(tidyverse)

download_data <- function(query) {
tb <- bq_project_query(Sys.getenv('GOOGLE_PROJECT'), query = query, default_dataset = Sys.getenv('WORKSPACE_CDR'))
bq_table_download(tb)
}

DATASET <- Sys.getenv('WORKSPACE_CDR')

df = download_data(str_glue("SELECT DISTINCT
person_id,
MIN(observation_date) AS primary_consent_date
FROM `{DATASET}.concept`
JOIN `{DATASET}.concept_ancestor` on concept_id = ancestor_concept_id
JOIN `{DATASET}.observation` on descendant_concept_id = observation_concept_id
WHERE concept_name = 'Consent PII' AND concept_class_id = 'Module'
GROUP BY 1"))
head(df)

In [ ]:
dim(df)